# Write a GPU kernel

This notebook compiles the Metal source you edit here on the iPad, dispatches it on the GPU, and compares its output with NumPy. This is a small numerical correctness fixture, not a performance benchmark.

In [ ]:
import numpy as np
from vaultlab import metal
source = r'''
#include <metal_stdlib>
using namespace metal;
kernel void compute(device const float* a [[buffer(0)]],
                    device const float* b [[buffer(1)]],
                    device float* out [[buffer(2)]],
                    uint i [[thread_position_in_grid]]) {
    out[i] = a[i] * a[i] + b[i];
}
'''
a = np.linspace(-2, 2, 1024, dtype=np.float32)
b = np.linspace(0, 1, 1024, dtype=np.float32)
result, receipt = metal.kernel(source, [a, b], len(a))
print(receipt)

In [ ]:
expected = a * a + b
np.testing.assert_allclose(result, expected, rtol=1e-6, atol=1e-6)
print(f"Verified {len(result)} GPU results against the CPU calculation.")

## Change the calculation

Modify the Metal expression and the NumPy reference together. Break one deliberately and confirm that the assertion detects the mismatch. Keep indexing within each buffer; a native GPU dispatch cannot be interrupted mid-kernel.